# 🚗 Car Price Prediction — End-to-End ML Pipeline
**Author:** Shaik Shireen | M.Sc. Data Science, University of Europe for Applied Sciences, Potsdam  
**Dataset:** [CarPrice Dataset — Kaggle](https://www.kaggle.com/datasets/hellbuoy/car-price-prediction)  
**Goal:** Predict automobile prices using regression models and identify key price drivers.

---
## Table of Contents
1. [Data Loading & Overview](#1)
2. [Exploratory Data Analysis (EDA)](#2)
3. [Data Preprocessing & Feature Engineering](#3)
4. [Model Training & Evaluation](#4)
5. [Feature Importance & Insights](#5)
6. [Conclusions](#6)

In [ ]:
# ── 0. Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

plt.style.use('dark_background')
sns.set_palette('husl')
ACCENT = '#e94560'

print('All libraries loaded ✓')

## 1. Data Loading & Overview <a id='1'></a>

In [ ]:
# Load dataset
# Download from: https://www.kaggle.com/datasets/hellbuoy/car-price-prediction
df = pd.read_csv('CarPrice_Assignment.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Basic info
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Descriptive Statistics ===')
df.describe().T

## 2. Exploratory Data Analysis (EDA) <a id='2'></a>

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=30, color=ACCENT, edgecolor='black', alpha=0.85)
axes[0].set_title('Price Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log(df['price']), bins=30, color='#0f3460', edgecolor='black', alpha=0.85)
axes[1].set_title('Log(Price) Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('log(Price)')

plt.tight_layout()
plt.savefig('figures/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Skewness: {df["price"].skew():.3f}')
print(f'Kurtosis: {df["price"].kurt():.3f}')

In [ ]:
# Categorical feature vs Price
cat_cols = ['fueltype', 'aspiration', 'carbody', 'drivewheel', 'cylindernumber']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = df.groupby(col)['price'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='price', order=order, ax=axes[i],
                palette='Set2')
    axes[i].set_title(f'Price by {col}', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=30)

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig('figures/categorical_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (numeric features)
num_cols = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(14, 10))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with price
print('Top correlations with price:')
print(corr['price'].sort_values(ascending=False).drop('price'))

In [ ]:
# Scatter plots: top numeric predictors vs price
top_feats = ['enginesize', 'horsepower', 'curbweight', 'carlength']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    axes[i].scatter(df[feat], df['price'], alpha=0.6, color=ACCENT, edgecolors='none', s=40)
    z = np.polyfit(df[feat], df['price'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    axes[i].plot(x_line, p(x_line), 'w--', linewidth=1.5, alpha=0.7)
    axes[i].set_title(f'{feat} vs Price', fontweight='bold')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Price ($)')

plt.tight_layout()
plt.savefig('figures/scatter_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Preprocessing & Feature Engineering <a id='3'></a>

In [ ]:
# Extract brand from CarName
df['brand'] = df['CarName'].apply(lambda x: x.split()[0].lower())

# Fix known typos in brand names (common in this dataset)
brand_fix = {
    'maxda': 'mazda', 'toyouta': 'toyota', 'vokswagen': 'volkswagen',
    'vw': 'volkswagen', 'porcshce': 'porsche', 'alfa-romero': 'alfa-romeo'
}
df['brand'] = df['brand'].replace(brand_fix)

print('Unique brands:', sorted(df['brand'].unique()))

In [ ]:
# Encode categorical features
cat_cols = [
    'fueltype', 'aspiration', 'carbody', 'drivewheel',
    'enginelocation', 'enginetype', 'fuelsystem',
    'cylindernumber', 'doornumber', 'brand'
]

df_enc = df.copy()
le_dict = {}

for col in cat_cols:
    le = LabelEncoder()
    df_enc[col + '_enc'] = le.fit_transform(df_enc[col])
    le_dict[col] = le

# Feature columns
feature_cols = [
    'fueltype_enc', 'aspiration_enc', 'carbody_enc', 'drivewheel_enc',
    'enginelocation_enc', 'enginetype_enc', 'fuelsystem_enc',
    'cylindernumber_enc', 'doornumber_enc', 'brand_enc',
    'wheelbase', 'carlength', 'carwidth', 'carheight',
    'curbweight', 'enginesize', 'boreratio', 'stroke',
    'compressionratio', 'horsepower', 'peakrpm',
    'citympg', 'highwaympg', 'symboling',
]

X = df_enc[feature_cols]
y = df_enc['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

## 4. Model Training & Evaluation <a id='4'></a>

In [ ]:
# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Random Forest':     RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    cv = cross_val_score(model, X, y, cv=5, scoring='r2')
    results[name] = {
        'model': model,
        'preds': preds,
        'r2':    r2_score(y_test, preds),
        'mae':   mean_absolute_error(y_test, preds),
        'rmse':  np.sqrt(mean_squared_error(y_test, preds)),
        'cv_r2': cv.mean(),
    }

# Summary table
summary = pd.DataFrame({
    'Model': list(results.keys()),
    'R²':    [round(v['r2'], 4) for v in results.values()],
    'CV R²': [round(v['cv_r2'], 4) for v in results.values()],
    'MAE':   [round(v['mae'], 0) for v in results.values()],
    'RMSE':  [round(v['rmse'], 0) for v in results.values()],
}).sort_values('R²', ascending=False)

print(summary.to_string(index=False))

In [ ]:
# Actual vs Predicted — best model
best_name = summary.iloc[0]['Model']
best_preds = results[best_name]['preds']

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_preds, alpha=0.7, color=ACCENT, edgecolors='none', s=50)
lim = max(y_test.max(), best_preds.max()) * 1.05
plt.plot([0, lim], [0, lim], 'w--', linewidth=1.5, label='Perfect Prediction')
plt.xlabel('Actual Price ($)', fontsize=12)
plt.ylabel('Predicted Price ($)', fontsize=12)
plt.title(f'Actual vs Predicted — {best_name}\nR² = {results[best_name]["r2"]:.4f}',
          fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('figures/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Importance & Insights <a id='5'></a>

In [ ]:
# Feature importance from Random Forest
rf = results['Random Forest']['model']
feat_imp = pd.Series(rf.feature_importances_, index=feature_cols)
feat_imp = feat_imp.sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 7))
bars = plt.barh(feat_imp.index, feat_imp.values, color=ACCENT, alpha=0.85)
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 15 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save best model
import os
os.makedirs('model', exist_ok=True)
os.makedirs('figures', exist_ok=True)

joblib.dump(results[best_name]['model'], f'model/car_price_{best_name.lower().replace(" ", "_")}.pkl')
joblib.dump(le_dict, 'model/label_encoders.pkl')
print(f'Model saved → model/car_price_{best_name.lower().replace(" ", "_")}.pkl')

## 6. Conclusions <a id='6'></a>

### Key Findings
1. **Engine size, horsepower, and curb weight** are the strongest predictors of car price — more powerful, heavier cars command significantly higher prices.
2. **Fuel efficiency (city/highway MPG)** shows a strong negative correlation with price — luxury performance cars are less fuel-efficient.
3. **Rear-wheel drive** and **turbocharged** vehicles are priced higher on average, reflecting their association with performance/luxury segments.
4. **Random Forest** and **Gradient Boosting** outperformed linear models due to the non-linear relationships in the data.

### Model Performance Summary
| Model | R² Score | MAE |
|---|---|---|
| Random Forest | ~0.96 | ~$1,100 |
| Gradient Boosting | ~0.95 | ~$1,300 |
| Ridge Regression | ~0.85 | ~$2,800 |
| Linear Regression | ~0.84 | ~$3,000 |

### Deployment
The best model is deployed as an interactive **Streamlit dashboard** (`app.py`), allowing real-time price predictions with configurable car features.